In [8]:
#importing 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pandas as pd
import pickle
import numpy as np

In [4]:
# ── Load processed data ───────────────────────────────────────
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

In [5]:
# ── Load vocab ────────────────────────────────────────────────
with open("../models/deep_nn/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

# ── Load feature list ─────────────────────────────────────────
with open("../models/meta_features.pkl", "rb") as f:
    all_meta = pickle.load(f)

print(f"✅ Loaded | Vocab size: {len(vocab)}")

✅ Loaded | Vocab size: 10000


In [6]:
# ── Hyperparameters ───────────────────────────────────────────
MAX_LEN  = 100
VOCAB_SIZE = len(vocab)
EMBED_DIM  = 128

def encode_text(text, vocab, max_len):
    tokens  = text.lower().split()[:max_len]
    indices = [vocab.get(t, 1) for t in tokens]
    # pad
    indices += [0] * (max_len - len(indices))
    return indices


In [10]:
# ── Dataset ───────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, df, vocab, max_len):
        self.texts  = [encode_text(t, vocab, max_len)
                       for t in df["review_text"]]
        self.meta   = df[all_meta].values.astype(np.float32)
        self.labels = df["label"].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text":  torch.tensor(self.texts[idx], dtype=torch.long),
            "meta":  torch.tensor(self.meta[idx],  dtype=torch.float),
            "label": torch.tensor(self.labels[idx],dtype=torch.long),
        }

train_dataset = ReviewDataset(syn_train, vocab, MAX_LEN)
val_dataset   = ReviewDataset(syn_val,   vocab, MAX_LEN)
test_dataset  = ReviewDataset(syn_test,  vocab, MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=32)
test_loader   = DataLoader(test_dataset,  batch_size=32)
